In [1]:
import numpy as np
import pandas as pd
from datetime import date
import collections
import datetime
import os
import xarray as xr
import csv

modelnames_OG = ['BCC-CSM2-MR', 'CAMS-CSM1-0', 'CESM2', 'CESM2-WACCM', 'EC-Earth3', 'EC-Earth3-Veg', 'FGOALS-f3-L', 
                 'GFDL-ESM4', 'INM-CM4-8','INM-CM5-0', 'MPI-ESM1-2-HR', 'MRI-ESM2-0', 'NorESM2-MM']

modelnames_py = ['BCC-CSM2-MR','CESM2','CESM2-WACCM','EC-Earth3','EC-Earth3-Veg','FGOALS-f3-L','GFDL-ESM4',
              'INM-CM4-8','INM-CM5-0','MPI-ESM1-2-HR','MRI-ESM2-0', 'NorESM2-MM']

modelnames_glo = ['BCC-CSM2-MR','CAMS-CSM1-0','CESM2','CESM2-WACCM','EC-Earth3','EC-Earth3-Veg','FGOALS-f3-L',
                  'GFDL-ESM4','INM-CM4-8','INM-CM5-0','MPI-ESM1-2-HR','MRI-ESM2-0', 'NorESM2-MM']

SSPs = ['ssp126','ssp245','ssp370','ssp585']

In [2]:
#All of our basins
basins_all = {'RHINE':'6242', 'RHONE':'6243','PO':'6241', 'DANUBE':'6202', 'TITICACA':'3912', 'SANTA':'3425', 
            'OCONA':'3418', 'MAJES':'3416', 'MAGDALENA':'3227', 'AMAZON':'3203', 'YELCHO':'3429', 
            'VALDIVIA':'3428', 'SERRANO':'3426', 'RAPEL':'3423', 'PUELO':'3422', 'PASCUA':'3420', 
            'PALENA':'3419', 'HUASCO':'3412', 'COPIAPO':'3409', 'CISNES':'3408', 'BIOBIO':'3405', 'BAKER':'3404',
            'AZOPARDO':'3403', 'AISEN':'3401', 'SANTA CRUZ':'3244', 'NEGRO':'3232', 'COLORADO':'3212', 
            'CHICO':'3209', 'TORNEALVEN':'6255', 'THJORSA':'6254', 'OLFUSA':'6237', 'LULEALVEN':'6227', 
            'KUBAN':'6223', 'KALIXALVEN':'6219', 'GLOMAA':'6213', 'DRAMSELVA':'6209', 'SVARTA':'6110', 
            'LAGARFLJOT':'6104', 'JOKULSA A FJOLLUM':'6101', 'CLUTHA':'5406', 'YUKON':'4435', 'TAKU':'4431', 
             'SUSITNA':'4430','STIKINE':'4428', 'SKEENA':'4427','SKAGIT':'4426','NUSHAGAK':'4418','NASS':'4416',
            'KUSKOKWIM':'4414','FRASER':'4410', 'COPPER':'4408', 'COLUMBIA':'4406', 'ALSEK':'4401', 'NELSON':'4125', 
              'MACKENZIE':'4123','COLVILLE':'4110', 'YSYK-KOL':'2919', 'UVS NUUR':'2918', 'TARIM HE':'2914', 
              'TALAS':'2913', 'LAKE BALKHASH':'2910','HAR US NUUR':'2909', 'CHUY':'2905', 'ARAL SEA':'2902', 
              'YELLOW RIVER':'2434', 'MEKONG':'2421', 'KAMCHATKA':'2413', 'SALWEEN':'2319', 'IRRAWADDY':'2310', 
              'INDUS':'2309', 'GANGES':'2306','BRAHMAPUTRA':'2302', 'OB':'2108', 'INDIGIRKA':'2103','YANGTZE' : '2433'}

#### GloGEM

In [64]:
#We are currently missing data for RGI regions 8 and 6 for GloGEM-hence 10 less basins
basins_glo = {'RHINE':'6242', 'RHONE':'6243','PO':'6241', 'DANUBE':'6202', 'TITICACA':'3912', 'SANTA':'3425', 
            'OCONA':'3418', 'MAJES':'3416', 'MAGDALENA':'3227', 'AMAZON':'3203', 'YELCHO':'3429', 
            'VALDIVIA':'3428', 'SERRANO':'3426', 'RAPEL':'3423', 'PUELO':'3422', 'PASCUA':'3420', 
            'PALENA':'3419', 'HUASCO':'3412', 'COPIAPO':'3409', 'CISNES':'3408', 'BIOBIO':'3405', 'BAKER':'3404',
            'AZOPARDO':'3403', 'AISEN':'3401', 'SANTA CRUZ':'3244', 'NEGRO':'3232', 'COLORADO':'3212', 
            'CHICO':'3209', 'KUBAN':'6223', 'CLUTHA':'5406', 'YUKON':'4435', 'TAKU':'4431', 'SUSITNA':'4430','STIKINE':'4428', 'SKEENA':'4427','SKAGIT':'4426','NUSHAGAK':'4418','NASS':'4416',
            'KUSKOKWIM':'4414','FRASER':'4410', 'COPPER':'4408', 'COLUMBIA':'4406', 'ALSEK':'4401', 'NELSON':'4125', 
              'MACKENZIE':'4123','COLVILLE':'4110', 'YSYK-KOL':'2919', 'UVS NUUR':'2918', 'TARIM HE':'2914', 
              'TALAS':'2913', 'LAKE BALKHASH':'2910','HAR US NUUR':'2909', 'CHUY':'2905', 'ARAL SEA':'2902', 
              'YELLOW RIVER':'2434', 'MEKONG':'2421', 'KAMCHATKA':'2413', 'SALWEEN':'2319', 'IRRAWADDY':'2310', 
              'INDUS':'2309', 'GANGES':'2306','BRAHMAPUTRA':'2302', 'OB':'2108', 'INDIGIRKA':'2103', 'YANGTZE' : '2433'}



In [65]:
import json
def select_glaciers_json(basin='all'):
    '''
    Select glaciers within a basin by MRBID from a json-file,
    which is stored in the data directory.

    Args:
    -----
    basin: str
        String of MRBID or 'all'.

    Returns:
    --------
    If basin is 'all' a list of all relevant glaciers is returned, for
    initiating glacier simulations. If basin is a MRBID the list of glaciers
    within that basin is returned.
    
    Copy of a function written by Erik Holmgren (2022) in holmgren_gha.utils
    '''

    # fpath = './data/rgi_ids_per_basin.json'
    fpath = '/Users/finnwimberly/Library/CloudStorage/GoogleDrive-fwimberly@middlebury.edu/My Drive/Lizz Research Stuff/rgi_ids_per_basin.json'  
    with open(fpath) as f:
        basin_dict = json.load(f)

    if basin.lower() != 'all':
        glacier_list = basin_dict[basin]
    else:
        glacier_list = list(itertools.chain.from_iterable(basin_dict.values()))

    return glacier_list

In [66]:
basin_gls = {}
regions_of_basin = {}
for basin, ID in basins_all.items():
    basin_gls[basin] = select_glaciers_json(ID)
    regions_of_basin[basin] = []  # Initialize an empty list for each basin
    for g, glacier in enumerate(basin_gls[basin]):
        region = int(basin_gls[basin][g][6:8])
        if region not in regions_of_basin[basin]:
            regions_of_basin[basin].append(region)

In [67]:
#Loading in area data
fpathglo = '/Users/finnwimberly/Library/CloudStorage/GoogleDrive-fwimberly@middlebury.edu/My Drive/Lizz Research Stuff/Runoff-intercomparison/GloGEM-output/'

RGIregionsGlo = ['RGI01-Alaska', 'RGI02-WesternCanada', 'RGI03', 'RGI04', 'RGI05', 'RGI10-NorthAsia'
                 , 'RGI11-CentralEurope','RGI12-Caucasus', 'RGI13', 'RGI14-SouthAsiaWest', 
                 'RGI15-SouthAsiaEast', 'RGI16-LowLatitudes', 'RGI17-SouthernAndes', 
                 'RGI18-NewZealand']

#Dummy list so RGI IDs align with list position
RGIregionsGloX = ['RGI01-Alaska', 'RGI02-WesternCanada', 'RGI03', 'RGI04', 'RGI05', 'x', 'x',
                 'x', 'x', 'RGI10-NorthAsia', 'RGI11-CentralEurope',
                 'RGI12-Caucasus', 'RGI13', 'RGI14-SouthAsiaWest', 'RGI15-SouthAsiaEast', 
                 'RGI16-LowLatitudes', 'RGI17-SouthernAndes', 'RGI18-NewZealand']

regional_fnames = ['alaska', 'westerncanada', 'arcticcanadaN', 'arcticcanadaS', 'greenland', 
                   'northasia', 'centraleurope','caucasus', 'centralasiaN', 'centralasiaW', 
                   'centralasiaS', 'lowlatitudes', 'southernandes', 'newzealand']

#Code to import all areas
# all_areas = {}
# for r, region in enumerate(RGIregionsGlo):
#     region_areas = []
#     for s, SSP in enumerate(SSPs):
#         model_areas = []
#         for m, model in enumerate(modelnames_glo):
#             temp_df = pd.read_csv(fpathglo + region + '/files/' + model  + '/' + SSP + '/' + regional_fnames[r] + '_Area_r1.dat', sep='\s+', index_col="ID")
#             model_areas.append(temp_df)
#         region_areas.append(model_areas)
#     all_areas[region] = region_areas

#Code to import single GCM/SSP areas (which are equivalent in yr 2000)
all_areas = {}
for r, region in enumerate(RGIregionsGlo):
    region_area = []
    temp_df = pd.read_csv(fpathglo + region + '/files/BCC-CSM2-MR/ssp585/' + regional_fnames[r] + '_Area_r1.dat', sep='\s+', index_col="ID")
    region_area.append(temp_df)
    all_areas[region] = region_area

In [68]:
def sum_basin_area(basin_RGI_list, area_data):
    # Create new list to match our RGI formatting
    new_basin_list = [int(str(x)[-5:]) for x in basin_RGI_list]
    
    # Filter new_basin_list to keep only the indexes present in the DataFrame
    new_basin_list = [x for x in new_basin_list if x in area_data.index]
    
    # Extract glaciers contained in the list from original df and create a new df
    new_df = area_data.loc[new_basin_list].copy()
    # print(new_df)
    
    # Sum the values of the glaciers within the basin
    summed_basin_area = new_df.sum()
    #print(summed_basin_runoff)
    
    return summed_basin_area

In [69]:
baseline_area = {}
for r, region in enumerate(RGIregionsGlo):
    baseline_area[region] = all_areas[region][0]

In [99]:
regional_gls = {}
for b, basins in enumerate(basins_all):
    regional_gls[basin] = {}
    for r, region in enumerate(RGIregionsGlo):
        
        for g, glacier in enumerate(basin_gls[basin]):
            region = int(basin_gls[basin][g][6:8])
            if region not in regions:
                regions.append(region)
        regional_gls[basin][regino] =

In [114]:
basin_area_sums_glo = {}
sum1 = {}
sum2 = {}
for b, basin in enumerate(basins_glo):
    if len(regions_of_basin[basin]) ==2:
        
        sum1[basin] = sum_basin_area(basin_gls[basin], baseline_area[RGIregionsGloX[(regions_of_basin[basin][0]-1)]])
        sum2[basin] = sum_basin_area(basin_gls[basin], baseline_area[RGIregionsGloX[(regions_of_basin[basin][1]-1)]])
        basin_area_sums_glo[basin] = sum2[basin]
   
    else:
        basin_area_sums_glo[basin] = sum_basin_area(basin_gls[basin], baseline_area[RGIregionsGloX[(regions_of_basin[basin][0]-1)]])

In [80]:
basin_area_sums_glo = {}
sum1 = {}
sum2 = {}
for b, basin in enumerate(basins_glo):
    if len(regions_of_basin[basin]) ==2:
        
        #sum1[basin] = sum_basin_area(basin_gls[basin], baseline_area[RGIregionsGloX[(regions_of_basin[basin][0]-1)]])
        sum2[basin] = sum_basin_area(basin_gls[basin], baseline_area[RGIregionsGloX[(regions_of_basin[basin][1]-1)]])
        basin_area_sums_glo[basin] = sum2[basin]
   
    else:
        basin_area_sums_glo[basin] = sum_basin_area(basin_gls[basin], baseline_area[RGIregionsGloX[(regions_of_basin[basin][0]-1)]])

In [115]:
#Creating our df
initial_basin_areas_glo = {}
for b, basin in enumerate(basins_glo):
    initial_basin_areas_glo[basin] = basin_area_sums_glo[basin][20]

area_df_glo = pd.DataFrame({'GloGEM': initial_basin_areas_glo})
area_df_glo.index.name = 'Basin'

In [117]:
area_df_glo

,GloGEM
Basin,
AISEN,154.120
ALSEK,5498.554
AMAZON,1402.745
ARAL SEA,13761.375
AZOPARDO,30.165
...,...
YANGTZE,432.913
YELCHO,238.940
YELLOW RIVER,177.422


In [ ]:
#Exporting as a CSV to use for larger basins where reading in both discharge and area crashes my laptop
output_dir = '/Users/finnwimberly/Desktop/Lizz Research/CSV Outputs/Area/All Shared Basins/'

fname = 'InitialAreas_GloGEM.csv'

# Define the full path of the output file
output_path = os.path.join(output_dir, fname)

# Save the DataFrame as CSV
area_df_glo.to_csv(output_path, header=True, index=True)

#### PyGEM

In [27]:
#Importing all runoff data, taking annual sum, and converting m^3 to km^3
import glob   #use glob to group files by filename similarities (in this case, SSP)

RGIregionsPy = ['01', '02', '03', '04', '05', '10', '11', '12', '13', '14', '15', '16', '17', '18']

fpathPy1 = '/Users/finnwimberly/Library/CloudStorage/GoogleDrive-fwimberly@middlebury.edu/My Drive/'
fpathPy2 = 'Lizz Research Stuff/Runoff-intercomparison/PyGEM/'

area_ds = {}

for r, region in enumerate(RGIregionsPy):
    fpath1 = '/{}/area_annual-ssp585/R{}_area_annual_c2_ba1_1set_2000_2100--'.format(region, region)
    file_pattern = f'{fpathPy1 + fpathPy2 + fpath1}*.nc'
    file_list = glob.glob(file_pattern)
    
    datasets = []  # Create an empty list for each SSP
    if file_list:
        for file in file_list: 
            with xr.open_dataset(file) as ds:
                ds = ds.glac_area_annual.load()
                datasets.append(ds)
    
        combined_ds = xr.concat(datasets, dim='glacier')  # Concatenate the datasets
        area_ds[region] = combined_ds * 1e-6  #m^2 to km^2


In [28]:
basin_gls = {}
for basin, ID in basins_all.items():
    basin_gls[basin] = select_glaciers_json(ID)

In [32]:
# Sorting into basins
basin_areas = {}
for basin, glacier_list in basin_gls.items():
    ## loop over them all, drop the irrelevant IDs, and concatenate the result
    basin_areas[basin] = {}
    for r, region in enumerate(RGIregionsPy):
        ds_list = []
        try:
            ds_filtered = area_ds[region].where(area_ds[region].RGIId.isin(glacier_list), drop=True)
            #print(ds_filtered)
            ds_list.append(ds_filtered)
        except ValueError: ## happens if there are no glaciers from this batch in the selected region
            continue
        basin_areas[basin][region] = xr.concat(ds_list, dim='glacier')

In [33]:
#Summing basins
basin_sums_py = {}
for basin, glacier_list in basin_gls.items():
    basin_sums_py[basin] = {}
    for r, region in enumerate(RGIregionsPy):
        basin_sums_py[basin][region] = basin_areas[basin][region].sum(dim='glacier')

In [47]:
basin_area_sums_py = {}

# Iterate through each basin
for basin, region_data in basin_sums_py.items():
    # Initialize the sum for the current basin
    basin_sum = None
    
    # Iterate through each region in the current basin
    for region, area_data in region_data.items():
        # If it's the first region encountered for this basin, set it as the initial sum
        if basin_sum is None:
            basin_sum = area_data
        else:
            # Otherwise, add the area_data to the current sum
            basin_sum += area_data
    
    # Store the combined basin sum in the new dictionary
    basin_area_sums_py[basin] = basin_sum

In [62]:
#Creating our df
#All models begin with same area so we just select the first one (BCC-CSM2-MR)
initial_basin_areas_py = {}
for b, basin in enumerate(basins_all):
    initial_basin_areas_py[basin] = basin_area_sums_py[basin].sel(year = 2000, model = 1).values

area_df_py = pd.DataFrame({'PyGEM': initial_basin_areas_py})
area_df_py.index.name = 'Basin'

In [63]:
area_df_py

,PyGEM
Basin,
AISEN,153.9163243842435
ALSEK,5499.486983319258
AMAZON,1400.1959543089079
ARAL SEA,12557.754489200304
AZOPARDO,30.207007901739672
...,...
YANGTZE,1710.2257143224706
YELCHO,238.95524261992426
YELLOW RIVER,177.38074014493043


#### OGGM

In [92]:
#Larger datasets so it helpful to break down into chunks
basins_europe = {'RHINE':'6242', 'RHONE':'6243','PO':'6241', 'DANUBE':'6202','TORNEALVEN':'6255', 
                 'THJORSA':'6254', 'OLFUSA':'6237', 'LULEALVEN':'6227', 'KUBAN':'6223', 
                 'KALIXALVEN':'6219', 'GLOMAA':'6213', 'DRAMSELVA':'6209', 'SVARTA':'6110', 
                'LAGARFLJOT':'6104', 'JOKULSA A FJOLLUM':'6101'}

basins_Namerica = {'YUKON':'4435', 'TAKU':'4431','SUSITNA':'4430','STIKINE':'4428', 
                   'SKEENA':'4427','SKAGIT':'4426','NUSHAGAK':'4418','NASS':'4416',
                   'KUSKOKWIM':'4414','FRASER':'4410', 'COPPER':'4408','COLUMBIA':'4406',
                    'ALSEK':'4401', 'NELSON':'4125', 'MACKENZIE':'4123','COLVILLE':'4110'}

basins_Samerica = {'TITICACA':'3912', 'SANTA':'3425', 
            'OCONA':'3418', 'MAJES':'3416', 'MAGDALENA':'3227', 'AMAZON':'3203', 'YELCHO':'3429', 
            'VALDIVIA':'3428', 'SERRANO':'3426', 'RAPEL':'3423', 'PUELO':'3422', 'PASCUA':'3420', 
            'PALENA':'3419', 'HUASCO':'3412', 'COPIAPO':'3409', 'CISNES':'3408', 'BIOBIO':'3405', 
            'BAKER':'3404','AZOPARDO':'3403', 'AISEN':'3401', 'SANTA CRUZ':'3244', 'NEGRO':'3232',
            'COLORADO':'3212', 'CHICO':'3209'}

basins_NZ = {'CLUTHA':'5406'}

basins_asia = {'YSYK-KOL':'2919', 'UVS NUUR':'2918', 'TARIM HE':'2914', 'TALAS':'2913', 
            'LAKE BALKHASH':'2910','HAR US NUUR':'2909', 'CHUY':'2905', 'ARAL SEA':'2902', 'YELLOW RIVER':'2434', 
          'MEKONG':'2421', 'KAMCHATKA':'2413', 'SALWEEN':'2319', 'IRRAWADDY':'2310', 'INDUS':'2309', 'GANGES':'2306',
          'BRAHMAPUTRA':'2302', 'OB':'2108', 'INDIGIRKA':'2103', 'YANGTZE' : '2433'}

In [86]:
#Generic filepath to navigate to Drive folder 
fpathOG1 = '/Users/finnwimberly/Library/CloudStorage/GoogleDrive-fwimberly@middlebury.edu/My Drive/'
fpathOG2 = 'Lizz Research Stuff/Runoff-intercomparison/OGGM/lschuster/runs_2023.3/output/basins/'

#Importing area data, OGGM is grouped by basin
#here we are just doing Namerica
#had trouble loading data all at once so breaking it up into separate cells by region
area_ds = {}
for basin, ID in basins_Namerica.items():
    fpath_basin = 'gcm_from_2000_bc_2000_2019/{}/'.format(ID)
    #print(f'{fpathOG + fpath_basin} (1)/*.nc')
    with xr.open_mfdataset(f'{fpathOG1 + fpathOG2 + fpath_basin}/*.nc') as ds:
        ds = ds.area.load()
    area_ds[basin] = ds

In [87]:
#Importing S America area data
for basin, ID in basins_Samerica.items():
    fpath_basin = 'gcm_from_2000_bc_2000_2019/{}/'.format(ID)
    #print(f'{fpathOG + fpath_basin} (1)/*.nc')
    with xr.open_mfdataset(f'{fpathOG1 + fpathOG2 + fpath_basin}/*.nc') as ds:
        ds = ds.area.load()
    area_ds[basin] = ds

In [88]:
#Importing Europe area data
for basin, ID in basins_europe.items():
    fpath_basin = 'gcm_from_2000_bc_2000_2019/{}/'.format(ID)
    #print(f'{fpathOG + fpath_basin} (1)/*.nc')
    with xr.open_mfdataset(f'{fpathOG1 + fpathOG2 + fpath_basin}/*.nc') as ds:
        ds = ds.area.load()
    area_ds[basin] = ds

In [89]:
#Importing NZ area data
for basin, ID in basins_NZ.items():
    fpath_basin = 'gcm_from_2000_bc_2000_2019/{}/'.format(ID)
    #print(f'{fpathOG + fpath_basin} (1)/*.nc')
    with xr.open_mfdataset(f'{fpathOG1 + fpathOG2 + fpath_basin}/*.nc') as ds:
        ds = ds.area.load()
    area_ds[basin] = ds

In [93]:
#Importing Asia area data
for basin, ID in basins_asia.items():
    fpath_basin = 'gcm_from_2000_bc_2000_2019/{}/'.format(ID)
    #print(f'{fpathOG + fpath_basin} (1)/*.nc')
    with xr.open_mfdataset(f'{fpathOG1 + fpathOG2 + fpath_basin}/*.nc') as ds:
        ds = ds.area.load()
    area_ds[basin] = ds

In [94]:
#Summing individual glacier runoff into basin totals and converting m^2 to km^2
basin_area_sums_OG = {}
for basin, ID in basins_all.items():
    basin_area_sums_OG[basin] = area_ds[basin].sum(dim = 'rgi_id') * 1e-6

In [95]:
#Creating our df
#All models begin with same area so we just select the first GCM (BCC-CSM2-MR) AND ssp126
initial_basin_areas_OG = {}
for b, basin in enumerate(basins_all):
    initial_basin_areas_OG[basin] = basin_area_sums_OG[basin].sel(time = 2000, gcm = 'BCC-CSM2-MR', scenario = 'ssp126').values

area_df_OG = pd.DataFrame({'OGGM': initial_basin_areas_OG})
area_df_OG.index.name = 'Basin'

In [96]:
area_df_OG

,OGGM
Basin,
AISEN,157.47366
ALSEK,5614.803
AMAZON,1490.8467
ARAL SEA,13309.696
AZOPARDO,31.184336
...,...
YANGTZE,1741.5363
YELCHO,247.73232
YELLOW RIVER,183.76598


### Creating Single DataFrame 
GloGEM lacks regions 8 and 6 (Scandinavia/Iceland) so we trim these ten basins from the PyGEM and OGGM dataframes

In [107]:
#Combining into a single dataframe 
#initial_areas_df = pd.concat([area_df_glo, area_df_py, area_df_OG], axis=1)
initial_areas_df = pd.concat([area_df_glo, area_df_py, area_df_OG], axis=1, join='inner')
initial_areas_df.loc['TARIM HE']

GloGEM             12096.616
PyGEM     26116.951624554284
OGGM                26528.58
Name: TARIM HE, dtype: object

In [ ]:
#Exporting as a CSV
output_dir = '/Users/finnwimberly/Desktop/Lizz Research/CSV Outputs/Area/All Shared Basins/'

fname = 'InitialAreas_ALL.csv'

# Define the full path of the output file
output_path = os.path.join(output_dir, fname)

# Save the DataFrame as CSV
initial_areas_df.to_csv(output_path, header=True, index=True)

### Errors explored:
It looks like we have uncovered the main source of intermodel runoff discrepancies! Here I'm going to calculate some differences and export the CSVs. We can then, more thoroughly and directly, examine the correlation between area and runoff in a separate notebook. 

In [ ]:
# Calculate the differences between each pair of models
initial_areas_df['GloGEM-PyGEM'] = abs(initial_areas_df['GloGEM'] - initial_areas_df['PyGEM'])
initial_areas_df['GloGEM-OGGM'] = abs(initial_areas_df['GloGEM'] - initial_areas_df['OGGM'])
initial_areas_df['PyGEM-OGGM'] = abs(initial_areas_df['PyGEM'] - initial_areas_df['OGGM'])

# Create a new DataFrame containing the largest differences for each basin and each model comparison
raw_diff_df = initial_areas_df[['GloGEM-PyGEM', 'GloGEM-OGGM', 'PyGEM-OGGM']]

In [ ]:
# Calculate the percent differences between each pair of models
initial_areas_df['GloGEM-PyGEM'] = (abs(initial_areas_df['GloGEM'] - initial_areas_df['PyGEM']) / initial_areas_df['GloGEM']) * 100
initial_areas_df['GloGEM-OGGM'] = (abs(initial_areas_df['GloGEM'] - initial_areas_df['OGGM']) / initial_areas_df['GloGEM']) * 100
initial_areas_df['PyGEM-OGGM'] = (abs(initial_areas_df['PyGEM'] - initial_areas_df['OGGM']) / initial_areas_df['PyGEM']) * 100

# Create a new DataFrame containing the largest percent differences for each basin and each model comparison
percent_diff_df = initial_areas_df[['GloGEM-PyGEM', 'GloGEM-OGGM', 'PyGEM-OGGM']]

In [ ]:
#Exporting as a CSVs
output_dir = '/Users/finnwimberly/Desktop/Lizz Research/CSV Outputs/Area/All Shared Basins/'

fname1 = 'RawDiff_AllInitialAreas.csv'
fname2 = 'PercentDiff_AllInitialAreas.csv'

# Define the full path of the output file
output_path1 = os.path.join(output_dir, fname1)
output_path2 = os.path.join(output_dir, fname2)

# Save the DataFrame as CSV
raw_diff_df.to_csv(output_path1, header=True, index=True)
percent_diff_df.to_csv(output_path2, header=True, index=True)

#### Outputting pre-summed area data as a single array to examine source of initial area differences

In [ ]:
#GloGEM

#We don't have a pre-summed dataset so we modify our summing function
def area_glaciers_in_basin(basin_RGI_list, area_data):
    # Create new list to match our RGI formatting
    new_basin_list = [int(str(x)[-4:]) for x in basin_RGI_list]
    
    # Filter new_basin_list to keep only the indexes present in the DataFrame
    new_basin_list = [x for x in new_basin_list if x in area_data.index]
    
    # Extract glaciers contained in the list from original df and create a new df
    new_df = area_data.loc[new_basin_list].copy()
    # print(new_df)
    
    return new_df

In [ ]:
basin_areas_glo = {}
initial_basin_areas_glo = {}
for b, basin in enumerate(basins_glo):
    basin_areas_glo[basin] = area_glaciers_in_basin(basin_gls[basin], baseline_area[RGIregionsGloX[(region_of_basin[basin]-1)]])
    initial_basin_areas_glo[basin] = basin_areas_glo[basin]['2000']

In [ ]:
#PyGEM
#Choose single GCM (BCC-CSM2-MR) to align with GloGEM
#These do not effect GloGEM initial area values but do SLIGHTLY within PyGEM
initial_basin_area = {}
initial_basin_area_py = {}
for b, basin in enumerate(basins_all):
    initial_basin_area[basin] = basin_areas[basin].sel(model=1, year=2000)
    rgi_id_index = initial_basin_area[basin].coords['RGIId']
    initial_basin_area_py[basin] = pd.DataFrame({'PyGEM': initial_basin_area[basin].values}, index=rgi_id_index)

In [ ]:
#OGGM
#Choose single SSP (585)and GCM to align with GloGEM
#GCMs and SSPs should NOT effect OGGM initial area values-though .equals returns false
#Presumably this is becuase of miniscule decimal differences
initial_basin_area = {}
initial_basin_area_OG = {}
for b, basin in enumerate(basins_all):
    initial_basin_area[basin] = area_ds[basin].sel(time=2000, scenario = 'ssp585', gcm = 'BCC-CSM2-MR')  * 1e-6 
    rgi_id_index = initial_basin_area[basin].coords['rgi_id']
    initial_basin_area_OG[basin] = pd.DataFrame({'OGGM': initial_basin_area[basin].values}, index=rgi_id_index)

In [ ]:
initial_basin_area_OG['COLORADO']